# Study Results Visualization

This notebook loads the repeated-run studies generated by `run_experiments.py`, splits results by homogeneous experimental setup, and compares the different learning approaches inside each setup.


In [ ]:
from pathlib import Path
import math

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from study_analysis import (
    GROUP_COLUMNS,
    DISPLAY_GROUP_COLUMNS,
    build_round_history,
    format_group_label,
    load_latest_study_dir,
    load_manifest,
    refresh_study_outputs,
)

OUTPUT_ROOT = Path("study_runs")
STUDY_DIR = None  # Example: Path("study_runs/study_20260501_153000")
FILTERS = {
    # "dataset": "mnist",
    # "model": "cnn",
    # "effective_num_clients": 20,
    # "effective_clients_per_round": 10,
}
SUMMARY_METRICS = [
    "Final_Accuracy",
    "Best_Accuracy",
    "Mean_Accuracy",
    "Mean_Duration_Sec",
    "Mean_TFLOPS",
    "Mean_Avg_Training_Time_Sec",
    "Mean_Avg_Communication_Time_Sec",
]
ROUND_METRICS = [
    "Accuracy",
    "Duration_Sec",
    "TFLOPS",
    "Avg_Training_Time_Sec",
    "Avg_Communication_Time_Sec",
]
MAX_GROUPS = None  # Set an integer to limit how many experiment groups are rendered.

study_dir = Path(STUDY_DIR) if STUDY_DIR else load_latest_study_dir(OUTPUT_ROOT)
manifest_df = load_manifest(study_dir)
run_summary_df, aggregate_df, pairwise_df = refresh_study_outputs(study_dir)
round_history_df = build_round_history(study_dir)

def apply_filters(frame: pd.DataFrame, filters: dict) -> pd.DataFrame:
    if frame is None or frame.empty:
        return frame
    filtered = frame.copy()
    for key, value in filters.items():
        if key not in filtered.columns:
            continue
        filtered = filtered[filtered[key] == value]
    return filtered

filtered_manifest_df = apply_filters(manifest_df, FILTERS)
filtered_run_summary_df = apply_filters(run_summary_df, FILTERS)
filtered_aggregate_df = apply_filters(aggregate_df, FILTERS)
filtered_pairwise_df = apply_filters(pairwise_df, FILTERS)
filtered_round_history_df = apply_filters(round_history_df, FILTERS)

print(f"Study directory: {study_dir.resolve()}")
print(f"Manifest rows: {len(manifest_df)} | Completed runs: {len(run_summary_df)}")
print(f"Filtered completed runs: {0 if filtered_run_summary_df is None else len(filtered_run_summary_df)}")


In [ ]:
def filter_by_group(frame: pd.DataFrame, group_values: dict) -> pd.DataFrame:
    if frame is None or frame.empty:
        return frame
    filtered = frame.copy()
    for key, value in group_values.items():
        if key not in filtered.columns:
            continue
        filtered = filtered[filtered[key] == value]
    return filtered

def available_experiment_groups(run_summary_frame: pd.DataFrame) -> pd.DataFrame:
    if run_summary_frame is None or run_summary_frame.empty:
        return pd.DataFrame()
    grouped = (
        run_summary_frame.groupby(GROUP_COLUMNS, dropna=False)
        .agg(
            num_runs=("run_id", "count"),
            num_approaches=("learning_type_display", "nunique"),
            approaches=("learning_type_display", lambda s: ", ".join(sorted(set(s)))),
        )
        .reset_index()
    )
    grouped["group_label"] = grouped.apply(lambda row: format_group_label(row.to_dict()), axis=1)
    return grouped[["group_label"] + GROUP_COLUMNS + ["num_runs", "num_approaches", "approaches"]]

group_catalog_df = available_experiment_groups(filtered_run_summary_df)
if group_catalog_df.empty:
    print("No experiment groups available for the selected filters.")
else:
    display(group_catalog_df.sort_values(DISPLAY_GROUP_COLUMNS).reset_index(drop=True))


In [ ]:
def plot_summary_metric_boxplots(group_runs_df: pd.DataFrame, metrics: list[str]) -> None:
    metrics = [metric for metric in metrics if metric in group_runs_df.columns]
    if not metrics:
        print("No summary metrics available for this group.")
        return
    ncols = 2
    nrows = math.ceil(len(metrics) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    approaches = sorted(group_runs_df["learning_type_display"].dropna().unique())
    for axis, metric in zip(axes, metrics):
        metric_df = group_runs_df[["learning_type_display", metric]].copy()
        metric_df[metric] = pd.to_numeric(metric_df[metric], errors="coerce")
        metric_df = metric_df.dropna()
        if metric_df.empty:
            axis.set_visible(False)
            continue
        data = []
        labels = []
        for approach in approaches:
            values = metric_df.loc[metric_df["learning_type_display"] == approach, metric].to_numpy(dtype=float)
            if values.size == 0:
                continue
            data.append(values)
            labels.append(approach)
        if not data:
            axis.set_visible(False)
            continue
        axis.boxplot(data, tick_labels=labels, showfliers=False)
        axis.set_title(metric)
        axis.tick_params(axis="x", rotation=25)
        axis.grid(False)
    for axis in axes[len(metrics):]:
        axis.set_visible(False)
    fig.tight_layout()
    plt.show()

def plot_round_metric_curves(group_round_df: pd.DataFrame, metrics: list[str]) -> None:
    metrics = [metric for metric in metrics if metric in group_round_df.columns]
    if not metrics:
        print("No round-level metrics available for this group.")
        return
    ncols = 2
    nrows = math.ceil(len(metrics) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]
    approaches = sorted(group_round_df["learning_type_display"].dropna().unique())
    for axis, metric in zip(axes, metrics):
        axis.set_title(metric)
        for approach in approaches:
            approach_df = group_round_df.loc[group_round_df["learning_type_display"] == approach, ["Round", metric]].copy()
            approach_df["Round"] = pd.to_numeric(approach_df["Round"], errors="coerce")
            approach_df[metric] = pd.to_numeric(approach_df[metric], errors="coerce")
            approach_df = approach_df.dropna()
            if approach_df.empty:
                continue
            stats = approach_df.groupby("Round")[metric].agg(["mean", "std", "count"]).reset_index()
            axis.plot(stats["Round"], stats["mean"], marker="o", label=approach)
            if (stats["count"] > 1).any():
                lower = (stats["mean"] - stats["std"].fillna(0.0)).to_numpy(dtype=float)
                upper = (stats["mean"] + stats["std"].fillna(0.0)).to_numpy(dtype=float)
                axis.fill_between(stats["Round"].to_numpy(dtype=float), lower, upper, alpha=0.15)
        axis.set_xlabel("Round")
        axis.grid(False)
        axis.legend()
    for axis in axes[len(metrics):]:
        axis.set_visible(False)
    fig.tight_layout()
    plt.show()

def render_experiment_group_report(group_values: dict) -> None:
    group_runs_df = filter_by_group(filtered_run_summary_df, group_values)
    group_aggregate_df = filter_by_group(filtered_aggregate_df, group_values)
    group_pairwise_df = filter_by_group(filtered_pairwise_df, group_values)
    group_round_df = filter_by_group(filtered_round_history_df, group_values)
    if group_runs_df is None or group_runs_df.empty:
        return

    display(Markdown(f"## {format_group_label(group_values)}"))

    run_columns = [
        "learning_type_display",
        "repetition",
        "Final_Accuracy",
        "Best_Accuracy",
        "Mean_Accuracy",
        "Mean_Duration_Sec",
        "Mean_TFLOPS",
        "Mean_Avg_Training_Time_Sec",
        "Mean_Avg_Communication_Time_Sec",
    ]
    available_run_columns = [column for column in run_columns if column in group_runs_df.columns]
    display(group_runs_df[available_run_columns].sort_values(["learning_type_display", "repetition"]).reset_index(drop=True))

    if group_aggregate_df is not None and not group_aggregate_df.empty:
        display(Markdown("### Descriptive Statistics"))
        display(group_aggregate_df.sort_values(["metric", "learning_type_display"]).reset_index(drop=True))

    if group_pairwise_df is not None and not group_pairwise_df.empty:
        display(Markdown("### Mann-Whitney U and A12"))
        display(group_pairwise_df.sort_values(["metric", "approach_a", "approach_b"]).reset_index(drop=True))

    display(Markdown("### Summary Metric Comparison Across Repetitions"))
    plot_summary_metric_boxplots(group_runs_df, SUMMARY_METRICS)

    if group_round_df is not None and not group_round_df.empty:
        display(Markdown("### Round-Level Comparison Across Learning Approaches"))
        plot_round_metric_curves(group_round_df, ROUND_METRICS)


In [ ]:
if filtered_run_summary_df is None or filtered_run_summary_df.empty:
    print("No completed runs available for the selected filters.")
else:
    grouped = filtered_run_summary_df.groupby(GROUP_COLUMNS, dropna=False)
    for group_index, (group_key, _) in enumerate(grouped, start=1):
        if MAX_GROUPS is not None and group_index > MAX_GROUPS:
            break
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        group_values = dict(zip(GROUP_COLUMNS, group_key))
        render_experiment_group_report(group_values)
